# Fairness-Aware GPT-2 — QQP TrainingTrains the three paraphrase models from the report and produces the numbers forTable 2. Scope is **QQP only** — that's where the fairness contribution is. SST,CFIMDB and sonnet generation are CS224N framework requirements with no fairnesscomponent and are skipped.**Before you run anything: Runtime → Change runtime type → T4 GPU.**| Run | Purpose | ~Time (5 epochs) ||---|---|---|| `cda_reg` | The deployed model. Do this first. | ~2 hr || `baseline` | Table 2 reference point | ~1 hr || `cda` | Table 2 middle row | ~1.3 hr |Free Colab disconnects when idle and caps sessions. Run **one at a time**, anddownload the results after each. Don't start all three and walk away.

In [ ]:
# Check we actually have a GPU. If this says "no GPU", stop and fix the runtime —# CPU training here would take weeks, not hours.!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU — Runtime > Change runtime type > T4 GPU"

## 1. Set upEdit `REPO` to your GitHub username, then run.

In [ ]:
REPO = "https://github.com/YOUR-USERNAME/fairness-aware-gpt2"   # <-- edit this!pip install -q uv!git clone -q $REPO fairness-repo%cd fairness-repo!uv sync --group train# Swap the CPU torch wheel (pinned for Streamlit Cloud) for the CUDA build.# --torch-backend=auto detects the driver, so this works on any Colab GPU.!uv pip install -q torch --torch-backend=auto!uv run python -c "import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())"

## 2. Get the QQP data`--train-size 283011` matches the pair count in the report (GLUE ships 363,846).It also cuts ~29% off every training run.The dev split is 40,430 either way — exactly the report's — so the fairnessmetrics stay comparable.

In [ ]:
!uv run scripts/download_data.py --only qqp --train-size 283011 --skip-test

## 3. Smoke test — do not skipTwo minutes. Catches every setup problem before you spend two hours on one.

In [ ]:
!uv run fairness-train --mode cda_reg \    --train data/quora-train.csv --dev data/quora-dev.csv \    --out /tmp/smoke --epochs 1 --train-subset 2000 --eval-subset 1000

If that printed a JSON line with `dev_acc`, everything works.## 4. Train `cda_reg` — the deployed model`--save-half` stores fp16 (~250MB instead of ~500MB), which matters forStreamlit Cloud's memory ceiling.**Keep this tab open and visible.** Colab kills idle sessions.

In [ ]:
!uv run fairness-train --mode cda_reg \    --train data/quora-train.csv --dev data/quora-dev.csv \    --out checkpoints/cda_reg --epochs 5 --lambda-fair 0.5 --save-half

## 5. Save the results before doing anything elseWrites `results/reproduced/cda_reg.json` — that's what makes your dashboard show**your** numbers next to the paper's. Download it now, in case the session diesduring the next run.

In [ ]:
from google.colab import files!zip -qr reproduced.zip results/reproducedfiles.download("reproduced.zip")

## 6. Push the model to Hugging FaceGet a **Write** token at huggingface.co/settings/tokens.

In [ ]:
!uv run huggingface-cli login

In [ ]:
HF_USER = "YOUR-HF-USERNAME"   # <-- edit this!uv run scripts/push_to_hub.py --ckpt checkpoints/cda_reg --repo $HF_USER/fairness-gpt2-qqp

Then set `MODEL_REPO = "YOUR-HF-USERNAME/fairness-gpt2-qqp"` in your Streamlitapp's Settings → Secrets. Live predictions turn on.---## 7. The other two models (optional, for full Table 2)Only worth it if you want the baseline/CDA comparison. `cda_reg` alone is enoughto deploy.Run these **one at a time**, re-downloading `reproduced.zip` after each.

In [ ]:
!uv run fairness-train --mode baseline \    --train data/quora-train.csv --dev data/quora-dev.csv \    --out checkpoints/baseline --epochs 5

In [ ]:
!uv run fairness-train --mode cda \    --train data/quora-train.csv --dev data/quora-dev.csv \    --out checkpoints/cda --epochs 5

In [ ]:
# Grab everything once all three are donefrom google.colab import files!zip -qr reproduced.zip results/reproducedfiles.download("reproduced.zip")

## What to compare againstAt **5 epochs**, the report's Table 4 gives you a direct comparison:| Mode | Dev acc | Subgroup gap | Flip rate ||---|---|---|---|| CDA | 0.8835 | 0.0433 | 0.0392 || CDA + Reg. | 0.8856 | 0.0512 | 0.0296 |Table 4 has no 5-epoch baseline, so compare that one against the 10-epoch row(0.8955 / 0.0365 / 0.0456) and expect it to land lower on accuracy.**Your flip rates will not match.** The report gives substitution *counts*(60/20/22) and three examples, not the lists — `identity.py` reconstructs them.A different lexicon means different counterfactuals. Accuracy should land close;flip rate is the number that moves.